# Stage 7 — The EXACT gsMap Pipeline, Run on YOUR Embedding — A1

**Inputs you attach to this notebook (Kaggle → Add Input):**
1. Your embedding file **`10x_human_lymph_node_A1_seed_42_embeddings.npy`**
   (uploaded by you; the notebook finds it by exact filename).
2. The A1 dataset (`spatial-multiomics-6datasets-private`) — gsMap still needs
   the raw counts, gene names and spatial coordinates from `adata_RNA.h5ad`;
   the embedding alone does not contain them.

**Design:** gsMap (Song et al., *Nature* 2025) normally trains its own 32-d GAT
embedding in Step 1. **We skip Step 1 entirely** and supply YOUR embedding
instead: stored in `obsm['fused']`, passed to `run_latent_to_gene`
via `--latent_representation fused` (the documented custom-latent route).
Everything downstream (microdomains → GSS → SNP annotation → per-spot S-LDSC
→ Cauchy per domain) is gsMap's own machinery, unchanged — the exact gsMap.

**Frozen inputs:** your A1 seed-42 embedding + domains from the
fixed one-pipeline recipe (standard-scale → k-means, declared k=10). The
embedding and domains are never retrained or re-selected using disease data.
A post-hoc ARI alignment check (~0.22 expected) verifies the embedding rows
line up with the A1 spots before anything runs.

**Pilot traits (ready-formatted in the official example bundle):**
- **Height** — NEGATIVE CONTROL. Expect: no domain significant.
- **IQ** — pipeline proof: a real trait producing maps + a domain table.
Phase 2 (SLE) is built in at the end — the GWAS is munged in-notebook to
LDSC format; only Steps 4–5 re-run per trait.

**v3 hardening (lessons from runs 1–2):** chunked verified downloads;
readiness judged by required FILES, not folders (a partial extraction from a
disk-full crash once slipped through and killed Step 4); tarballs deleted
immediately after extraction (20 GB disk); completed gsMap steps are skipped
via gsMap's own output markers; every gsMap call prints its own error output
on failure; the chromosome loop checkpoints and resumes.
**v4 fix (2026-11-02):** the required resource/GWAS files are located by NAME
anywhere in the extracted tree and every gsMap path is derived from the finds
— a live run extracted cleanly yet failed the old fixed-path check. An
incomplete tree is moved aside and re-fetched once, never silently deleted.
**v5 fix (2026-11-02):** the 1000G panel is per-chromosome
(`1000G.EUR.QC.1.bim` … `.22.bim`) — no merged `.bim` exists, and v3/v4 keyed
the readiness check on that phantom file. Discovery now keys on the
chromosome-1 file and derives the `--bfile_root` prefix from it.

**Phase 2 (2026-11-03):** SLE appended at the end — Bentham 2015
(GCST003156, EUR, N=23,210), munged in-notebook to LDSC format and filtered
to gsMap's HapMap3 keep-lists; LD scores reused; the result is printed and
pushed as a calibration comparison against the Height background.

**Output:** `runs/s7-disease/` (Cauchy domain × trait tables + run log).


In [ ]:
import os, sys, json, subprocess, time

for pkg in ["gsMap", "scanpy", "anndata", "scikit-learn", "pandas", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

r = subprocess.run(["gsmap", "-v"], capture_output=True, text=True)
print("gsmap version:", (r.stdout or r.stderr).strip()[:100])

import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

print("imports ok")


In [ ]:
# ---------- GitHub: clone + push helper (results -> runs/s7-disease) ----------
REPO = "/kaggle/working/my_research"
REPO_URL = "https://github.com/rifatahsanpul0k/my_research.git"
def sh(*args):
    return subprocess.run(list(args), capture_output=True, text=True, cwd=REPO)
os.makedirs(REPO, exist_ok=True)
OUT = os.path.join(REPO, "runs", "s7-disease")
os.makedirs(OUT, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    import base64, shutil
    _sec = UserSecretsClient()
    _tok = _sec.get_secret('GITHUB_TOKEN')
    _b64 = base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    _GIT = ["git", "-c", f"http.extraHeader=Authorization: Basic {_b64}"]
    if not os.path.isdir(os.path.join(REPO, ".git")):
        if os.listdir(REPO):
            bak = REPO + "_stalebak"
            if os.path.exists(bak):
                shutil.rmtree(bak)
            shutil.move(REPO, bak)
            os.makedirs(REPO, exist_ok=True)
        subprocess.run(_GIT + ["clone", REPO_URL, REPO], check=True)
        sh("git", "config", "user.name", "kaggle-runner")
        sh("git", "config", "user.email", "kaggle-runner@local")
    os.makedirs(OUT, exist_ok=True)
    print("git ready")
except Exception as e:
    print("git setup issue:", str(e)[:200])

def git_push(msg):
    try:
        os.makedirs(OUT, exist_ok=True)
        sh("git", "add", "runs/s7-disease")
        r = sh("git", "diff", "--cached", "--quiet")
        if r.returncode == 0:
            print("nothing new to push")
            return
        sh("git", "commit", "-m", msg)
        subprocess.run(_GIT + ["pull", "--rebase", "origin", "main"],
                       capture_output=True, text=True, cwd=REPO)
        subprocess.run(_GIT + ["push", "origin", "main"],
                       capture_output=True, text=True, cwd=REPO)
        print("pushed")
    except Exception as e:
        print("push failed:", str(e)[:300])


In [ ]:
# ---------- Load A1 data + YOUR uploaded embedding; recompute domains ----------
import glob as _glob
MARKER = "10x_human_lymph_node_A1"
def find_data_root():
    for base in ["/kaggle/input/spatial-multiomics-6datasets-private",
                 "/kaggle/working/spatial-multiomics-6datasets-private",
                 "/kaggle/working"]:
        if os.path.isdir(os.path.join(base, MARKER)):
            return base
    hits = [os.path.dirname(os.path.dirname(p)) for p in
            _glob.glob(f"/kaggle/input/**/{MARKER}/adata_RNA.h5ad", recursive=True)]
    return hits[0] if hits else None

DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d",
                    "pulokpulok/spatial-multiomics-6datasets-private",
                    "-p", "/kaggle/working", "--unzip"], check=True)
    DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)

root = os.path.join(DATA_ROOT, MARKER)
adata = sc.read_h5ad(os.path.join(root, "adata_RNA.h5ad"))
adata.var_names_make_unique()
print("adata_RNA:", adata.shape, "| obsm keys:", list(adata.obsm.keys()))
assert "spatial" in adata.obsm, "obsm['spatial'] missing — cannot package for gsMap"

# YOUR embedding: found by exact filename wherever Kaggle mounted the upload
EMB_NAME = "10x_human_lymph_node_A1_seed_42_embeddings.npy"
emb_hits = (_glob.glob(f"/kaggle/input/**/{EMB_NAME}", recursive=True) +
            _glob.glob(f"/kaggle/working/**/{EMB_NAME}", recursive=True))
assert emb_hits, (f"{EMB_NAME} not found under /kaggle/input or /kaggle/working — "
                  "attach your uploaded embedding dataset to this notebook (Add Input)")
EMB_PATH = sorted(emb_hits)[0]
print("embedding file:", EMB_PATH)
E = np.load(EMB_PATH).astype(np.float32)
print(f"embedding: {E.shape}")
assert E.ndim == 2, f"embedding must be 2-D (n_spots x dims), got shape {E.shape}"
if E.shape[0] != adata.shape[0] and E.shape[1] == adata.shape[0]:
    print("WARNING: embedding looks transposed — transposing to (n_spots, dims)")
    E = E.T
assert E.shape[0] == adata.shape[0], "row mismatch: embedding vs adata — ALIGNMENT BROKEN"

# Domains by the FIXED one-pipeline recipe (standard-scale + kmeans, declared k)
K_A1 = 10
Ez = StandardScaler().fit_transform(E)
domains = KMeans(n_clusters=K_A1, n_init=10, random_state=42).fit_predict(Ez)
print("domain sizes:", np.bincount(domains))

# Plumbing check ONLY (post-hoc, quarantined): ARI vs manual annotation.
# Exp0 reported BASE/A1 standard/kmeans ~ 0.221. A wildly different value here
# means the embedding rows do not align with the adata rows. Labels select nothing.
gt = pd.read_csv(os.path.join(root, "annotation.csv"), index_col=0)
ari = adjusted_rand_score(np.asarray(gt["manual-anno"]).astype(str), domains)
print(f"ALIGNMENT CHECK — post-hoc ARI = {ari:.3f} (expect ~0.22)")
assert ari > 0.10, "ARI far below expected — likely row misalignment. STOP."


In [ ]:
# ---------- Package the gsMap input h5ad ----------
# gsMap format (official docs): counts in layers['count'], coords in
# obsm['spatial'], annotation as an obs column, latent in obsm under our key.
pkg = adata.copy()
counts = pkg.X.toarray() if hasattr(pkg.X, "toarray") else np.asarray(pkg.X)
pkg.X = counts.astype(np.float32)
pkg.layers["count"] = counts.astype(np.float32)
pkg.obsm["fused"] = E
pkg.obs["domain"] = pd.Categorical([f"D{d}" for d in domains])

GSMAP_DIR = "/kaggle/working/gsmap"
os.makedirs(GSMAP_DIR, exist_ok=True)
H5AD = os.path.join(GSMAP_DIR, "A1_input.h5ad")
pkg.write_h5ad(H5AD)
print("wrote", H5AD)
print("layers:", list(pkg.layers.keys()), "| obsm:", list(pkg.obsm.keys()),
      "| obs domain counts:", pkg.obs["domain"].value_counts().to_dict())


In [ ]:
# ---------- Download gsMap resources + example GWAS (discovered by filename) ----------
import glob, shutil

def run(cmd, **kw):
    print("+", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {' '.join(cmd)}")
    return r

def run_v(cmd):
    # Like run(), but captures output and prints gsMap's own last 30 lines on
    # failure — a bare wrapper traceback hid the real error in run 1.
    print("+", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        tail = "\n".join((r.stdout + "\n" + r.stderr).splitlines()[-30:])
        print("----- gsMap output (last 30 lines) -----")
        print(tail)
        print("----------------------------------------")
        raise RuntimeError(f"failed ({r.returncode}): {' '.join(cmd[:2])}")
    return r

def fetch(url, dest, chunk=32 * 1024 * 1024, tries=6):
    # Chunked Range download. One long stream to this server keeps dying:
    # wget truncated silently; curl -C - resume appended to corrupt prefixes.
    # Each 32 MB chunk is a short request, retried on its own, appended in
    # order. Re-running resumes — but only from a prefix THIS function wrote
    # (marker file); foreign partials are discarded. gzip -t verifies before
    # tar ever touches the file.
    import urllib.request
    def _req(range_hdr=None):
        h = {"User-Agent": "Mozilla/5.0"}
        if range_hdr:
            h["Range"] = range_hdr
        return urllib.request.Request(url, headers=h)
    with urllib.request.urlopen(_req(), timeout=60) as r:
        total = int(r.headers.get("Content-Length") or 0)
    print(f"  {dest}: expected {total/1e9:.2f} GB", flush=True)
    assert total > 0, "server gave no Content-Length"
    marker = dest + ".chunked"
    if os.path.exists(dest) and not os.path.exists(marker):
        print("  discarding partial file from an earlier download method")
        os.remove(dest)
    open(marker, "a").close()
    have = os.path.getsize(dest) if os.path.exists(dest) else 0
    if have > total:
        os.remove(dest); have = 0
    with open(dest, "ab") as f:
        pos = have
        while pos < total:
            end = min(pos + chunk, total) - 1
            data = None
            for a in range(tries):
                try:
                    with urllib.request.urlopen(_req(f"bytes={pos}-{end}"),
                                                timeout=180) as resp:
                        if resp.status == 200:
                            raise IOError("server ignored the Range header")
                        data = resp.read()
                    if len(data) == end - pos + 1:
                        break
                    print(f"  chunk @{pos/1e9:.2f} GB short read, retry", flush=True)
                    data = None
                except Exception as e:
                    print(f"  chunk @{pos/1e9:.2f} GB failed ({str(e)[:80]}), "
                          f"retry {a+1}/{tries}", flush=True)
                    data = None
            if data is None:
                raise RuntimeError(f"chunk at byte {pos} failed {tries} times")
            f.write(data); f.flush()
            pos += len(data)
            if (pos // chunk) % 10 < 1:
                print(f"  {pos/1e9:.2f} / {total/1e9:.2f} GB", flush=True)
    ok = subprocess.run(["gzip", "-t", dest]).returncode == 0
    print(f"  gzip integrity: {'OK' if ok else 'BAD'}", flush=True)
    if not ok:
        os.remove(dest); os.remove(marker)
        raise RuntimeError("archive corrupt even after chunked download: " + dest)
    os.remove(marker)

def df():
    print(subprocess.run(["df", "-h", "/kaggle/working"],
                         capture_output=True, text=True).stdout)

# --- Resources are DISCOVERED by filename, never trusted at fixed paths ---
# (2026-11-02 live failure: the bundle downloaded + extracted cleanly, yet
# v3's fixed-path readiness assert fired. So the pipeline now FINDS the files
# it needs wherever they sit in the extracted tree, derives every downstream
# gsMap path from the finds, and never deletes an extracted tree on a failed
# check — an incomplete tree is moved aside and re-fetched exactly once.)
RES = os.path.join(GSMAP_DIR, "gsMap_resource")  # expected root; discovery may find another

def _find1(pat):
    hits = sorted(glob.glob(os.path.join(GSMAP_DIR, "**", pat), recursive=True))
    return hits[0] if hits else None

def _tree(p, depth=2):
    for dp, dns, fns in os.walk(p):
        lvl = dp.replace(p, "").count(os.sep)
        if lvl > depth:
            continue
        print("  " * lvl + os.path.basename(dp) + "/")
        if lvl == depth:
            for fn in sorted(fns)[:8]:
                print("  " * (lvl + 1) + fn)

BIM = BIM22 = GTF_FILE = W1 = HEIGHT_SS = IQ_SS = None
BFILE_ROOT = HM_ROOT = W_PREFIX = None

def discover():
    global BIM, BIM22, GTF_FILE, W1, HEIGHT_SS, IQ_SS, BFILE_ROOT, HM_ROOT, W_PREFIX, RES
    # The 1000G panel is PER-CHROMOSOME: 1000G.EUR.QC.1.bim ... .22.bim.
    # There is NO single merged 1000G.EUR.QC.bim — keying the search on that
    # name was the v3/v4 bug (the bundle was complete all along; the check
    # could never pass). gsMap's --bfile_root takes the prefix and appends
    # .<chrom> itself, so BFILE_ROOT is derived from the chromosome-1 file.
    BIM = _find1("1000G.EUR.QC.1.bim")
    BIM22 = _find1("1000G.EUR.QC.22.bim")
    GTF_FILE = _find1("gencode*.gtf") or _find1("*.gtf")
    W1 = _find1("weights.1.l2.ldscore.gz")
    HEIGHT_SS = _find1("GIANT_EUR_Height_2022_Nature.sumstats.gz")
    IQ_SS = _find1("IQ_NG_2018.sumstats.gz")
    if BIM:
        BFILE_ROOT = BIM[:-len(".1.bim")]
        RES = os.path.dirname(os.path.dirname(os.path.dirname(BIM)))
    if W1:
        W_PREFIX = W1[:-len("1.l2.ldscore.gz")]  # .../weights_hm3_no_hla/weights.
    hm_dirs = sorted(glob.glob(os.path.join(GSMAP_DIR, "**", "hapmap3_snps"),
                                recursive=True))
    if hm_dirs:
        HM_ROOT = os.path.join(hm_dirs[0], "hm")
    return bool(BIM and GTF_FILE and W1)

def resources_found():
    discover()
    return bool(BIM and BIM22 and GTF_FILE and W1 and HM_ROOT)

def example_found():
    discover()
    return bool(HEIGHT_SS and IQ_SS)

def ensure_bundle(kind, url, tarball, std_dirname, found_fn):
    if found_fn():
        print(f"{kind}: already present and verified by discovery")
        return
    std = os.path.join(GSMAP_DIR, std_dirname)
    aside = None
    if os.path.isdir(std):
        aside = std + "_incomplete"
        if os.path.exists(aside):
            shutil.rmtree(aside)
        os.rename(std, aside)
        print(f"{std_dirname} present but key files not found — moved aside, re-fetching")
    fetch(url, tarball)
    run(["tar", "-xzf", tarball])
    os.remove(tarball)  # disk discipline: 20 GB working dir
    if not found_fn():
        print(f"--- tree of {GSMAP_DIR} (depth 2) ---")
        _tree(GSMAP_DIR)
        raise RuntimeError(f"{kind} files not found even after fresh extraction "
                           "— send the tree listing above")
    if aside and os.path.isdir(aside):
        shutil.rmtree(aside)
        print(f"fresh {kind} bundle verified — removed the incomplete copy")

os.chdir(GSMAP_DIR)
df()
ensure_bundle("resource",
              "https://yanglab.westlake.edu.cn/data/gsMap/gsMap_resource.tar.gz",
              "gsMap_resource.tar.gz", "gsMap_resource", resources_found)
ensure_bundle("example",
              "https://yanglab.westlake.edu.cn/data/gsMap/gsMap_example_data.tar.gz",
              "gsMap_example_data.tar.gz", "gsMap_example_data", example_found)
# Only the GWAS folder of the example bundle is needed — drop the example ST data
if HEIGHT_SS:
    ex_dir = os.path.dirname(os.path.dirname(HEIGHT_SS))
    if os.path.isdir(os.path.join(ex_dir, "ST")):
        shutil.rmtree(os.path.join(ex_dir, "ST"))
        print("dropped example ST data (not needed)")
for leftover in glob.glob("*.tar.gz"):
    os.remove(leftover)
    print("removed leftover archive:", leftover)
df()

print("resource root :", RES)
print("bfile prefix  :", BFILE_ROOT)
print("bim chrom 1   :", BIM)
print("bim chrom 22  :", BIM22)
print("gtf           :", GTF_FILE)
print("keep_snp root :", HM_ROOT)
print("weights prefix:", W_PREFIX)
print("height gwas   :", HEIGHT_SS)
print("iq gwas       :", IQ_SS)
assert resources_found() and example_found(), "resource check failed — see paths above"
print("all resources verified")


In [ ]:
# ---------- gsMap Step 2: latent_to_gene with OUR embedding ----------
# (Their Step 1 — training a GAT embedding — is SKIPPED by design.)
WD = os.path.join(GSMAP_DIR, "workdir")
os.makedirs(WD, exist_ok=True)
MKSCORE = os.path.join(WD, "A1", "latent_to_gene", "A1_gene_marker_score.feather")
if os.path.exists(MKSCORE):
    print("mkscore feather found — latent_to_gene already done, skipping")
else:
    t0 = time.time()
    run_v(["gsmap", "run_latent_to_gene",
           "--workdir", WD,
           "--sample_name", "A1",
           "--input_hdf5_path", H5AD,
           "--annotation", "domain",
           "--latent_representation", "fused"])
    print(f"latent_to_gene done in {(time.time()-t0)/60:.1f} min")


In [ ]:
# ---------- gsMap Step 3: generate LD scores (resumable, per chromosome) ----------
# Skipped entirely when gsMap's own .done marker exists (it writes one only
# after ALL chromosomes finish). Otherwise a progress JSON checkpoints each
# finished chromosome, and a failed chromosome retries once at 100
# spots/chunk (memory guard; docs quote ~40 GB at 1000, A1 has ~3.5k spots).
DONE_MARKER = os.path.join(WD, "A1", "generate_ldscore", "A1_generate_ldscore.done")
if os.path.exists(DONE_MARKER):
    print("gsMap .done marker found — LD scores already complete, skipping Step 3")
else:
    assert os.path.exists(MKSCORE), "mkscore feather missing — Step 2 did not finish"
    PROG = os.path.join(GSMAP_DIR, "ldscore_progress.json")
    done = set(json.load(open(PROG))) if os.path.exists(PROG) else set()
    print("chromosomes already done:", sorted(done))
    df()

    def ldscore_cmd(chrom, spc):
        return ["gsmap", "run_generate_ldscore",
                "--workdir", WD,
                "--sample_name", "A1",
                "--chrom", str(chrom),
                "--bfile_root", BFILE_ROOT,
                "--keep_snp_root", HM_ROOT,
                "--gtf_annotation_file", GTF_FILE,
                "--gene_window_size", "50000",
                "--spots_per_chunk", str(spc)]

    t0 = time.time()
    for chrom in range(1, 23):
        if chrom in done:
            continue
        tc = time.time()
        try:
            run_v(ldscore_cmd(chrom, 250))
        except RuntimeError:
            print(f"  chrom {chrom} failed at 250 spots/chunk — retrying at 100")
            run_v(ldscore_cmd(chrom, 100))
        done.add(chrom)
        json.dump(sorted(done), open(PROG, "w"))
        print(f"  chrom {chrom} done in {(time.time()-tc)/60:.1f} min "
              f"({len(done)}/22)", flush=True)
    print(f"LD scores complete in {(time.time()-t0)/60:.1f} min this run")
    df()


In [ ]:
# ---------- gsMap Step 4: per-spot spatial LDSC (Height control, then IQ) ----------
TRAITS = {
    "Height": HEIGHT_SS,
    "IQ":     IQ_SS,
}
for trait, ss in TRAITS.items():
    assert os.path.exists(ss), f"sumstats missing for {trait}: {ss}"
print("sumstats verified for:", list(TRAITS))
for trait, ss in TRAITS.items():
    t0 = time.time()
    run_v(["gsmap", "run_spatial_ldsc",
           "--workdir", WD,
           "--sample_name", "A1",
           "--trait_name", trait,
           "--sumstats_file", ss,
           "--w_file", W_PREFIX,
           "--num_processes", "4"])
    print(f"spatial_ldsc [{trait}] done in {(time.time()-t0)/60:.1f} min", flush=True)


In [ ]:
# ---------- gsMap Step 5: Cauchy combination per domain ----------
for trait in TRAITS:
    run_v(["gsmap", "run_cauchy_combination",
           "--workdir", WD,
           "--sample_name", "A1",
           "--trait_name", trait,
           "--annotation", "domain"])
print("cauchy done")

# Discover + collect outputs (exact filenames are workdir-internal; glob them)
found = sorted(glob.glob(os.path.join(WD, "**", "*auchy*"), recursive=True) +
               glob.glob(os.path.join(WD, "**", "*A1*"), recursive=True))
for f in found:
    print(f)


In [ ]:
# ---------- Summarise: domain x trait table -> push to GitHub ----------
summary = {"dataset": "A1", "embedding": "uploaded 10x_human_lymph_node_A1_seed_42_embeddings.npy -> obsm['fused']",
           "domains": "standard+kmeans k=10 (fixed one-pipeline recipe)",
           "traits": {}, "notes": "Height = negative control (expect null)"}

cauchy_files = [f for f in glob.glob(os.path.join(WD, "**", "*.csv*"), recursive=True)
                if "auchy" in os.path.basename(f).lower()]
print("cauchy files:", cauchy_files)
for f in cauchy_files:
    try:
        dfx = pd.read_csv(f)
        print("\n", os.path.basename(f)); print(dfx.to_string()[:1500])
        dest = os.path.join(OUT, "A1_" + os.path.basename(f))
        shutil.copy(f, dest)
    except Exception as e:
        print("could not read", f, str(e)[:150])

with open(os.path.join(OUT, "A1_gsmap_pilot_log.json"), "w") as fh:
    json.dump({"cauchy_files": [os.path.basename(f) for f in cauchy_files],
               "alignment_ari_posthoc": float(ari)}, fh, indent=2)

git_push("s7-disease: exact gsMap A1 (uploaded seed42 embedding; Height control + IQ)")
print("\nPilot done. Read the Height table first: it should be null across domains.")
df()


## Phase 2 — SLE (first disease trait)

**GWAS:** Bentham et al. 2015 (*Nat Genet*), GWAS Catalog **GCST003156** —
European ancestry, 7,219 cases + 15,991 controls (N = 23,210), matching the
1000G EUR LD panel. The raw deposit (`chrom, pos, rsid, other_allele,
effect_allele, p, beta, se`) is munged in the next cell into LDSC format
(`SNP, A1, A2, Z = beta/se, N`) and filtered to the HapMap3 SNP set gsMap
itself scores — the same treatment the example Height/IQ files received.

Only `spatial_ldsc` + Cauchy run; the pilot's LD scores are reused (they are
trait-independent). **Read SLE against the Height table, not against
p = 0.05** — a domain counts only where SLE clearly exceeds its Height
background. The final cell prints and pushes that calibration comparison.


In [ ]:
# ---------- Phase 2: SLE sumstats - download + munge to LDSC format ----------
import glob, gzip, os, subprocess, time
import numpy as np
import pandas as pd

GSMAP_DIR = "/kaggle/working/gsmap"
TRAIT_DIR = os.path.join(GSMAP_DIR, "traits")
os.makedirs(TRAIT_DIR, exist_ok=True)
SLE_RAW = os.path.join(TRAIT_DIR, "bentham_2015_sle.tsv.gz")
SLE_SS = os.path.join(TRAIT_DIR, "SLE.sumstats.gz")
SLE_URL = ("https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/"
           "GCST003001-GCST004000/GCST003156/"
           "bentham_2015_26502338_sle_efo0002690_1_gwas.sumstats.tsv.gz")
SLE_BYTES = 323169910          # Content-Length verified 2026-11-03
SLE_N = 7219 + 15991           # Bentham 2015 cases + controls = 23210

def _get(url, dest, expected):
    import urllib.request
    got = os.path.getsize(dest) if os.path.exists(dest) else 0
    if got != expected:
        if os.path.exists(dest):
            os.remove(dest)
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as r, open(dest, "wb") as f:
            while True:
                chunk = r.read(1 << 22)
                if not chunk:
                    break
                f.write(chunk)
        got = os.path.getsize(dest)
    assert got == expected, f"size mismatch: got {got}, expected {expected}"
    subprocess.run(["gzip", "-t", dest], check=True)
    print("downloaded + verified:", dest, f"({got/1e6:.0f} MB)")

if os.path.exists(SLE_SS):
    print("SLE.sumstats.gz already present - skipping download + munge")
else:
    _get(SLE_URL, SLE_RAW, SLE_BYTES)
    t0 = time.time()
    raw = pd.read_csv(SLE_RAW, sep="\t", usecols=["rsid", "other_allele",
                        "effect_allele", "beta", "se"], dtype={"rsid": str})
    print(f"raw rows: {len(raw):,}")
    m = pd.DataFrame({
        "SNP": raw["rsid"],
        "A1": raw["effect_allele"].str.upper(),
        "A2": raw["other_allele"].str.upper(),
        "Z": raw["beta"] / raw["se"],
        "N": SLE_N})
    ok = (m["SNP"].str.startswith("rs")
          & m["A1"].isin(list("ACGT")) & m["A2"].isin(list("ACGT"))
          & np.isfinite(m["Z"]))
    m = m[ok].drop_duplicates("SNP")
    amb = {("A", "T"), ("T", "A"), ("C", "G"), ("G", "C")}
    m = m[[(a1, a2) not in amb for a1, a2 in zip(m["A1"], m["A2"])]]
    print(f"after QC (rsID, ACGT alleles, finite Z, dedup, non-ambiguous): {len(m):,}")
    # keep only the SNPs gsMap scored: its own HapMap3 keep-lists
    keep = set()
    for f in glob.glob(os.path.join(GSMAP_DIR, "gsMap_resource", "LDSC_resource",
                                    "hapmap3_snps", "*")):
        if not os.path.isfile(f):
            continue
        try:
            fh = gzip.open(f, "rt") if f.endswith(".gz") else open(f)
            with fh:
                keep.update(tok.split()[0] for tok in fh if tok.startswith("rs"))
        except Exception:
            pass
    if keep:
        m = m[m["SNP"].isin(keep)]
        print(f"after HapMap3 keep-list filter: {len(m):,}")
    else:
        print("WARNING: no HapMap3 keep-lists found - keeping all QC'd SNPs")
    assert len(m) > 200_000, "suspiciously few SNPs left after munging"
    m.to_csv(SLE_SS, sep="\t", index=False, compression="gzip")
    os.remove(SLE_RAW)   # disk discipline
    print(f"wrote {SLE_SS} in {time.time()-t0:.0f}s | Z range "
          f"[{m['Z'].min():.1f}, {m['Z'].max():.1f}]")
print(subprocess.run(["df", "-h", "/kaggle/working"],
                     capture_output=True, text=True).stdout)


In [ ]:
# ---------- Phase 2: spatial_ldsc + Cauchy for SLE (LD scores reused) ----------
import glob, os, subprocess, time

GSMAP_DIR = "/kaggle/working/gsmap"
WD = os.path.join(GSMAP_DIR, "workdir")
DONE = os.path.join(WD, "A1", "generate_ldscore", "A1_generate_ldscore.done")
assert os.path.exists(DONE), ("LD scores not found - run the Phase 1 cells first "
                              "(Steps 2-3, ~40 min); they are reused for every trait")
_w1 = sorted(glob.glob(os.path.join(GSMAP_DIR, "**", "weights.1.l2.ldscore.gz"),
                       recursive=True))
assert _w1, "LDSC weights not found under the gsmap dir"
W_PREFIX = _w1[0][:-len("1.l2.ldscore.gz")]
print("weights prefix:", W_PREFIX)
SLE_SS = os.path.join(GSMAP_DIR, "traits", "SLE.sumstats.gz")
assert os.path.exists(SLE_SS), "SLE.sumstats.gz missing - run the munge cell above"

def runp(cmd):
    print("+", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout or "")[-2500:])
        print((r.stderr or "")[-2500:])
        raise RuntimeError("command failed: " + cmd[1])
    return r

t0 = time.time()
runp(["gsmap", "run_spatial_ldsc", "--workdir", WD, "--sample_name", "A1",
      "--trait_name", "SLE", "--sumstats_file", SLE_SS,
      "--w_file", W_PREFIX, "--num_processes", "4"])
print(f"spatial_ldsc [SLE] done in {(time.time()-t0)/60:.1f} min")
runp(["gsmap", "run_cauchy_combination", "--workdir", WD, "--sample_name", "A1",
      "--trait_name", "SLE", "--annotation", "domain"])
print("cauchy [SLE] done")


In [ ]:
# ---------- Phase 2: calibrated read (SLE vs Height background) + push ----------
import glob, os, shutil, subprocess
import numpy as np
import pandas as pd

WD = os.path.join("/kaggle/working/gsmap", "workdir")

def _cauchy(trait):
    fs = glob.glob(os.path.join(WD, "**", f"A1_{trait}.Cauchy.csv.gz"), recursive=True)
    assert fs, f"cauchy output for {trait} not found"
    return pd.read_csv(fs[0])

tab = None
for trait in ["Height", "IQ", "SLE"]:
    d = _cauchy(trait)[["annotation", "p_cauchy"]].rename(columns={"p_cauchy": trait})
    tab = d if tab is None else tab.merge(d, on="annotation")
for trait in ["Height", "IQ", "SLE"]:
    tab[trait + " -log10p"] = -np.log10(tab[trait].clip(lower=1e-300))
tab["SLE minus Height (log10)"] = tab["SLE -log10p"] - tab["Height -log10p"]
tab = tab.sort_values("SLE minus Height (log10)", ascending=False)
pd.set_option("display.width", 200)
print(tab.to_string(index=False))
print("\nRead SLE by the LAST column: domains where SLE clearly exceeds its "
      "Height background are the candidates. Raw SLE p-values alone prove "
      "nothing - Height was significant almost everywhere.")

OUT = os.path.join("/kaggle/working/my_research", "runs", "s7-disease")
os.makedirs(OUT, exist_ok=True)
sle_f = glob.glob(os.path.join(WD, "**", "A1_SLE.Cauchy.csv.gz"), recursive=True)[0]
shutil.copy(sle_f, os.path.join(OUT, "A1_A1_SLE.Cauchy.csv.gz"))
tab.to_csv(os.path.join(OUT, "A1_calibration_comparison.csv"), index=False)
try:
    def _sh(*a):
        return subprocess.run(list(a), capture_output=True, text=True,
                              cwd="/kaggle/working/my_research")
    _sh("git", "add", "runs/s7-disease")
    _sh("git", "commit", "-m", "stage7 phase2: SLE cauchy + calibration comparison (A1)")
    _sh("git", "pull", "--rebase")
    r = _sh("git", "push")
    print("pushed" if r.returncode == 0 else
          "push failed (tables saved in runs/s7-disease):\n" + r.stderr[-500:])
except Exception as e:
    print("push skipped:", e)


## Phase 2 — how it continues

Identical three cells per additional trait (new sumstats file, same munge
rules, same LD scores): **RA, MS, IBD, T1D, asthma, Hodgkin, DLBCL, CLL** —
the pre-registered lymph-node panel.

Gates before any disease claim: (1) the trait's domain ranking differs from
Height's; (2) the signal lands in domains that marker genes name as immune;
(3) the pattern survives the gsMap-own-embedding sensitivity run; (4) D1
replication. Wording: *disease-relevant programme*, never *diseased domain*.
